# Python Day 10 — Libraries, File Handling
**Class date:** 4 October 2026  
**Primary topics:** `os` library, paths, file handling, file modes, file cursors, reading, writing, and modifying text files.

> These notes consolidate the supplied class notebook, recording transcript, text-file examples, Python scripts, and assignment PDFs. The class notebook is the main source for the demonstrated code. The recording supplies explanations and practical context. The supplied notes PDF appears image-based and has no extractable text layer.

## Learning map
1. Use `os` to inspect the working directory and build portable paths.
2. Open files and understand file objects and file cursors.
3. Read whole files, one line at a time, or into a list of lines.
4. Write and append content.
5. Understand `r`, `w`, `a`, `r+`, `w+`, and `a+`.
6. Use `tell()`, `seek()`, and `truncate()` to understand and modify file content.
7. Read external data into Python and write application logs/configuration.

## 1. Libraries and `import`

A library is reusable code written by developers. It can contain one or more Python files and provide functions that we can use rather than implementing everything ourselves.

Python does not automatically load every installed library into every program. `import os` tells Python that this file needs the `os` library, so its functionality can be accessed from the current file.

`os` is part of Python's standard library, so it is included with a normal Python installation; you do not install it with `pip install os`.

In [ ]:
import os

### Calling a function from a library

Use the library name, a dot, and then the function name:

```python
os.getcwd()
```

Without `os.`, Python would look for a function named `getcwd` in the current program's available names.

## 2. Working directories and file paths

### `os.getcwd()`

Returns the **current working directory**: the folder from which the current Python process is working. It returns the directory path, not the filename of the current script.

```python
os.getcwd()
```

### `os.listdir()`

Returns the names of files and folders in a directory.

- `os.listdir()` uses the current working directory by default.
- `os.listdir(path)` lists the contents of the directory supplied as `path`.

```python
os.listdir()
os.listdir("C:/some/project/folder")
```

The path passed to `listdir` must identify a directory that exists.

In [ ]:
# Inspect the current working directory
current_folder = os.getcwd()
print(current_folder)

# List files and folders in the current working directory
items_here = os.listdir()
print(items_here)

### `getcwd()` versus `listdir()`

| Function | What it answers |
|---|---|
| `os.getcwd()` | Where am I working? |
| `os.listdir()` | What files/folders are inside this location? |
| `os.listdir(path)` | What files/folders are inside that specified location? |

These functions have different jobs: one returns a directory path; the other returns directory entries.

### Path separators across operating systems

The class discussed these common conventions:

- Windows commonly displays paths with backslashes, for example `C:\\Users\\Name\\project`.
- macOS and Linux use forward slashes, for example `/Users/name/project`.
- Windows also accepts forward-slash paths in many Python path operations.

When writing paths manually, choosing the correct separators can become error-prone and can make code less portable. Prefer `os.path.join()` for combining path components.

In [ ]:
# Example: build a path from the current directory and a filename
file_name = "a.txt"
file_path = os.path.join(os.getcwd(), file_name)
print(file_path)

### `os.path.join()`

`os.path.join()` combines path components using the separator appropriate for the operating system.

It solves two problems highlighted in class:

1. You do not need to decide manually whether to use `/` or `\\`.
2. You do not need to manually concatenate directory and filename strings.

It builds a path string; it does **not** create the file.

```python
path = os.path.join(os.getcwd(), "a.txt")
```

Compare that with manually concatenating strings:

```python
path = os.getcwd() + "/a.txt"
```

The second version is ordinary string concatenation and requires you to manage separators yourself.

In [ ]:
# Compare the idea: joining path parts versus manually concatenating strings
folder = os.getcwd()
filename = "a.txt"

joined_path = os.path.join(folder, filename)
print(joined_path)

## 3. File handling: the basic idea

File handling lets a Python program store and retrieve information outside the running program.

Examples discussed in class:

- Read text or data from files such as `.txt`, `.csv`, `.json`, or spreadsheet files.
- Keep configuration or prompt text outside the Python source code.
- Read data into Python so that it can be processed.
- Write logs or generated information to files.
- Create files programmatically.

The key distinction: a `.py` file contains instructions; a `.txt`, `.csv`, or other data file can contain the data those instructions need to load. We generally **read/load** a data file; we do not execute a CSV file as though it were a Python program.

## 4. Opening a file and understanding the file object

The built-in `open()` function opens a file and returns a file object. The file object gives access to operations such as reading, writing, checking the mode, and closing the file.

A basic read operation:

```python
f = open(file_path, "r")
```

- `file_path` identifies the file.
- `"r"` means read mode.
- The file must already exist when opened in `"r"` mode.

The class inspected these file-object attributes:

- `f.name` — the file name/path associated with the object.
- `f.mode` — the mode used to open the file.
- `f.closed` — whether the file has been closed.

After opening a file, close it when finished. A closed file cannot be used for further input/output operations until it is opened again.

In [ ]:
# Use a file named a.txt in the current working directory.
# This assumes a.txt exists in that folder.
file_path = os.path.join(os.getcwd(), "a.txt")
f = open(file_path, "r")

print(f.name)
print(f.mode)
print(f.closed)

f.close()
print(f.closed)

### Reading the entire file: `read()`

`f.read()` reads the remaining content from the current cursor position to the end of the file and returns it as one string.

Important cursor behaviour:

- When a file is first opened for reading, the cursor starts at the beginning.
- After `read()` consumes the entire file, the cursor is at the end of the file (EOF).
- Calling `read()` again at EOF returns an empty string (`''`), because there is no remaining content to read.
- To read again from the beginning, reopen the file or move the cursor with `seek(0)` where the mode permits it.

In [ ]:
file_path = os.path.join(os.getcwd(), "a.txt")
f = open(file_path, "r")

content = f.read()
print(content)
print("Characters:", len(content))
print("Words:", len(content.split()))

# The cursor is already at EOF, so this returns an empty string.
again = f.read()
print(repr(again))

f.close()

### `read()` memory consideration

`read()` loads the remaining file content into one Python string. For a very large file, that can require substantial memory.

The class contrasted this with `readline()`, which lets a program process one line at a time. That is often more suitable when a file is very large and you do not need the entire contents in memory at once.

## 5. Reading one line at a time: `readline()`

`f.readline()` reads the next line from the current cursor position and advances the cursor to the next line.

- Each call reads one line.
- Repeated calls move through the file.
- Once there are no lines left, it returns `''`.
- The returned line commonly includes its newline character (`\n`) when one exists in the file.

This makes it possible to process a large text file incrementally.

In [ ]:
file_path = os.path.join(os.getcwd(), "a.txt")
f = open(file_path, "r")

line_1 = f.readline()
line_2 = f.readline()
line_3 = f.readline()
line_4 = f.readline()  # Empty string if there is no fourth line

print(repr(line_1))
print(repr(line_2))
print(repr(line_3))
print(repr(line_4))

f.close()

### Loop through a file with `readline()`

The class demonstrated a `while` loop that keeps reading until `readline()` returns an empty string.

```python
while operation_not_completed:
    content = f.readline()
    if content != '':
        print(content)
    else:
        operation_not_completed = False
```

The empty string is the end-of-file signal in this pattern.

In [ ]:
file_path = os.path.join(os.getcwd(), "b.txt")
f = open(file_path, "r")

operation_not_completed = True
while operation_not_completed:
    content = f.readline()

    if content != "":
        print(content, end="")
    else:
        operation_not_completed = False

f.close()

### Why might the output contain extra blank lines?

A line read from a text file may already end with `\n`. The `print()` function also adds a newline by default. Printing the line without controlling `end` can therefore produce an additional visible blank line.

The class compared:

```python
s1 = "hello\n"
s2 = "hello"
print(s1)
print(s2)
```

- `\n` is a newline character inside the string.
- `print()` adds its own newline by default.
- `print(content, end="")` avoids adding another newline after the content.

This is also a useful connection to the `print()` parameters covered in the previous class.

In [ ]:
s1 = "hello\n"
s2 = "hello"

print("First string:")
print(s1)
print("Second string:")
print(s2)

print("Controlled output:")
print(s1, end="")
print(s2)

## 6. Reading all lines: `readlines()`

`f.readlines()` reads all remaining lines and returns them as a **list of strings**.

- `read()` returns one string containing the remaining file content.
- `readline()` returns the next single line as a string.
- `readlines()` returns a list containing the remaining lines.

`readlines()` still loads all lines into memory. It is useful when you want a list of lines, but it is not the same memory strategy as processing a very large file one line at a time.

In [ ]:
file_path = os.path.join(os.getcwd(), "a.txt")
f = open(file_path, "r")

all_lines = f.readlines()
print(all_lines)
print(type(all_lines))

f.close()

## 7. Extracting text between square brackets

### Class exercise

The supplied `b.txt` contains lines such as:

```text
hello this is a dummy file. [line-1]
hello this is a dummy file. [line-2]
hello this is a dummy file. [line-3]
I am learning [line-4] data science from krish naik academy.
```

Task: read the file line by line and print only the content between `[` and `]`.

The recording showed two approaches: `find()` with slicing, and `split()`.

### Approach A — `find()` and slicing

- `find("[")` returns the index of the opening bracket.
- Add `1` to start immediately after `[`.
- `find("]")` returns the index of the closing bracket.
- Slice from the start index up to (but not including) the closing bracket index.

In [ ]:
sample_line = "hello this is a dummy file. [line-1]"

start = sample_line.find("[") + 1
end = sample_line.find("]")

result = sample_line[start:end]
print(result)

### Approach B — `split()`

First split on the opening bracket and take the part after it (`[-1]`). Then split that result on the closing bracket and take the part before it (`[0]`).

This is the class's compact expression:

```python
content.split("[")[-1].split("]")[0]
```

In [ ]:
sample_line = "I am learning [line-4] data science from krish naik academy."

inside_brackets = sample_line.split("[")[-1].split("]")[0]
print(inside_brackets)

### Complete file-processing example

The following combines file reading, a `while` loop, the empty-string EOF check, and bracket extraction. It assumes each relevant line contains a pair of square brackets, as in the supplied class sample.

In [ ]:
file_path = os.path.join(os.getcwd(), "b.txt")
f = open(file_path, "r")

operation_not_completed = True
while operation_not_completed:
    line = f.readline()

    if line != "":
        start = line.find("[") + 1
        end = line.find("]")
        print(line[start:end])
    else:
        operation_not_completed = False

f.close()

## 8. Writing to a file: `write()` and mode `"w"`

Open a file in `"w"` mode to write content.

- If the file does not exist, `"w"` creates it.
- If the file already exists, opening it in `"w"` truncates it: the previous content is removed.
- `write()` writes the supplied string.
- `write()` does not automatically add a newline. Include `\n` when you want a line break.
- Close the file after writing.

**Important:** Opening an existing file with `"w"` can erase its previous content immediately, even before you call `write()`.

In [ ]:
file_path = os.path.join(os.getcwd(), "c.txt")

f = open(file_path, "w")
f.write("hello this is file - 2")
f.close()

### Repeated `write()` calls and newlines

Without `\n`, multiple writes are placed directly next to each other. The supplied `c.txt` demonstrates this:

```python
f.write("hello [line-1]")
f.write("hello [line-2]")
```

The result is concatenated on one line. Use `\n` if you want separate lines.

In [ ]:
file_path = os.path.join(os.getcwd(), "c.txt")

f = open(file_path, "w")
f.write("hello [line-1]\n")
f.write("hello [line-2]\n")
f.close()

### Writing a Python script as text

The original notebook demonstrated creating `a.py` by writing Python source code into a file:

```python
f.write("for i in range(0,10):\n")
f.write("  print(i)")
```

This shows that a program can generate another file. The write operation stores text; it does not execute the generated script.

In [ ]:
file_path = os.path.join(os.getcwd(), "generated_example.py")

f = open(file_path, "w")
f.write("for i in range(0, 10):\n")
f.write("    print(i)")
f.close()

## 9. File modes: read, write, append, and plus modes

The class compared six modes. The table below captures the behaviours explained during the session.

| Mode | Read? | Write? | Creates if missing? | Clears existing content on open? | Main behaviour |
|---|---:|---:|---:|---:|---|
| `r` | Yes | No | No | No | Read an existing file |
| `w` | No | Yes | Yes | Yes | Start fresh and write |
| `a` | No | Yes | Yes | No | Add content at the end |
| `r+` | Yes | Yes | No | No | Read and modify existing content |
| `w+` | Yes | Yes | Yes | Yes | Start fresh, with read and write access |
| `a+` | Yes | Yes | Yes | No | Read and append; writes go to the end |

**Remember:** `+` adds the other capability (read or write) to the base mode. It does not remove the base mode's behaviour. In particular, `a+` still appends writes at EOF, even after you seek elsewhere.

### `r+` — read and write existing content

- The file must already exist.
- It does not clear the file when opened.
- It allows both reading and writing.
- Writing starts at the current cursor position and can overwrite existing characters.
- To write at the end, move the cursor to EOF (for example, by reading the remaining content or seeking to the end).

The recording emphasized that writing at the beginning with `r+` replaces characters at that position; it does not automatically shift all existing content to make room.

In [ ]:
# Demonstration: r+ overwrites from the current cursor position.
# Use a disposable file because this changes its contents.
demo_path = os.path.join(os.getcwd(), "rplus_demo.txt")

f = open(demo_path, "w")
f.write("hello world")
f.close()

f = open(demo_path, "r+")
f.write("HEY")
f.close()

f = open(demo_path, "r")
print(f.read())  # HEYlo world
f.close()

### `w+` — write and read

- Creates the file if it does not exist.
- Clears an existing file when opened.
- Allows both writing and reading.
- The cursor begins at the start of the newly emptied file.

The recording described this as a fresh start with read access also available.

### `a` and `a+` — append modes

- Append mode preserves existing content.
- Writes are placed at the end of the file.
- The file is created if it does not exist.
- `a+` adds read capability to append mode.
- With `a+`, the cursor initially starts at EOF. To read from the beginning, use `seek(0)` first.
- Even if you seek to the beginning, writes in append mode still go to the end; it is not a general-purpose modification mode.

In [ ]:
# Demonstration of a+.
# This creates the file if needed and appends new text.
demo_path = os.path.join(os.getcwd(), "append_demo.txt")

f = open(demo_path, "a+")
f.write("First line\n")
f.write("Second line\n")

# Move cursor to the beginning so that reading can retrieve the content.
f.seek(0)
print(f.read())

f.close()

## 10. `write()` and `writelines()`

- `write(string)` writes one string and returns the number of characters written.
- `writelines(iterable_of_strings)` writes each string from the iterable.
- `writelines()` does **not** insert newline characters automatically. Include `\n` in each string if separate lines are required.

Example:

```python
f.writelines(["line-3\n", "line-4\n", "line-5\n"])
```

In [ ]:
demo_path = os.path.join(os.getcwd(), "writelines_demo.txt")

f = open(demo_path, "w")
written_count = f.write("Hello\n")
print("Characters written:", written_count)

f.writelines(["line-2\n", "line-3\n", "line-4\n"])
f.close()

## 11. File cursor: `tell()` and `seek()`

### `tell()`

Returns an integer indicating the current file cursor position. The recording described it as a position measured in bytes or characters depending on the file/text handling context; do not assume it is always identical to Python's `len()` for every encoding or file type.

### `seek(offset)`

Moves the file cursor to a specified offset. In the examples in class, `seek(0)` moves the cursor to the beginning of the file.

A useful mental model:

- `tell()` asks: where is the cursor now?
- `seek(0)` says: move the cursor to the beginning.
- After reading to EOF, the cursor is at the end.

In [ ]:
demo_path = os.path.join(os.getcwd(), "cursor_demo.txt")

f = open(demo_path, "w")
f.write("Hello\n")
f.write("Hello again\n")
print("Position after writes:", f.tell())
f.close()

f = open(demo_path, "a+")
print("Position after opening in a+:", f.tell())

f.seek(0)
print("Position after seek(0):", f.tell())
print(f.read())

f.close()

### Why did `read()` return an empty string in `a+`?

When a file is opened in append mode, the initial cursor is at EOF. A call to `read()` at EOF has no remaining content to return. Move the cursor to the beginning first:

```python
f.seek(0)
content = f.read()
```

## 12. Modifying part of a file with `r+`, `seek()`, and `truncate()`

The recording walked through replacing the `el` in `hello` with `xy` (or another replacement) using a careful sequence.

Suppose the file contains:

```text
hello, how are you
```

Goal: remove the characters at positions 1 and 2 (`e` and `l`) while keeping the rest.

### Conceptual sequence

1. Open the file in `r+`.
2. `seek(1)` to position the cursor just after the initial `h`.
3. Read the remaining content into a variable.
4. Use string slicing to discard the first two characters from that remainder.
5. `seek(1)` again to return to the point where the replacement content should begin.
6. Call `truncate()` to remove the content from the current cursor position onward.
7. Write the retained content back.
8. Close the file.

`truncate()` removes content from the specified/current position onward; the class example uses it after moving the cursor to the point where the old tail should be removed.

In [ ]:
# Safe, reproducible demonstration using a disposable file.
demo_path = os.path.join(os.getcwd(), "modify_demo.txt")

f = open(demo_path, "w")
f.write("hello, how are you")
f.close()

f = open(demo_path, "r+")

f.seek(1)                 # Position after 'h'
content = f.read()        # Read the remaining text: 'ello, how are you'
content = content[2:]     # Discard 'e' and 'l'

f.seek(1)                 # Return to the position after 'h'
f.truncate()              # Remove the old content from here onward
f.write(content)          # Write the retained remainder

f.close()

f = open(demo_path, "r")
print(f.read())           # hlo, how are you
f.close()

### Important distinction: overwrite versus replace

A normal `write()` in `r+` overwrites from the current cursor position. It does not automatically remove the remaining old characters or shift them to the right.

To replace a section while preserving the rest, the demonstrated approach reads the remaining text, uses Python string operations to construct the desired content, truncates the old tail, and writes the new tail.

The recording noted that full text-editor-style modification is more involved because a program must account for indexes, newlines, spaces, and the position of the target text. The class's goal was to understand the underlying file operations, not build a complete editor.

## 13. Reading external data into Python

A key reason for reading files is to bring information stored outside the Python program into Python so that it can be processed.

The class gave examples such as:

- Read a prompt from a `.txt` file and pass that string to an LLM API.
- Read assignment text and count occurrences of words or patterns.
- Read values from a text or CSV data file before analysis or machine learning.
- Keep data/configuration separate from the code so a teammate can update the data without changing the Python logic.

The recording distinguished loading data from executing a file: a Python script reads a data file, then processes the values.

### Class example: read comma-separated values and convert them to integers

The supplied `variables.txt` contains:

```text
1,2,3,4,5,6,7,8,9,10,11,100,100
```

The original notebook:
1. Builds the path using `os.path.join()`.
2. Opens the file.
3. Reads the content as a string.
4. Splits the string on commas to get a list of strings.
5. Converts each list element to an integer.

In [ ]:
file_path = os.path.join(os.getcwd(), "variables.txt")
f = open(file_path, "r")

content = f.read()
f.close()

content = content.split(",")

for i in range(len(content)):
    content[i] = int(content[i])

print(content)

### Separating the data-loading logic into a function

The class then moved the loading logic into a function and returned the processed list. This keeps the data-reading code separate from the code that uses the data.

In [ ]:
import os

def get_input_from_txt():
    file_path = os.path.join(os.getcwd(), "variables.txt")
    f = open(file_path, "r")

    content = f.read()
    f.close()

    content = content.split(",")

    for i in range(len(content)):
        content[i] = int(content[i])

    return content

data = get_input_from_txt()

print(data)
print(len(data))

### Why keep data separate from code?

If values are stored in a separate text file, a teammate can update the data without editing the Python program. The Python code reads the latest file content when it runs.

This is the idea of a decoupled solution: the program's instructions and the data/configuration are stored separately.

## 14. Writing files in real projects: logs and generated configuration

The recording explained two practical reasons for writing files.

### Logs

A program can write information about its activity to a file, for example:

- Input received by a model or application.
- Output or prediction produced.
- Errors or unusual behaviour.
- Events that developers may need to inspect later.

Keeping logs allows a team to review what happened after the program has run, rather than relying only on information that was temporarily in RAM.

### Generated files and configuration

Programs and tools can create files automatically. The recording demonstrated a command-line project initialization workflow creating project files and configuration. The lesson: software often writes files on behalf of the user; file writing is not limited to manually creating a text document.

### Prompt files

A prompt can be stored in a text file and loaded by Python. This makes it possible to update the prompt without editing the Python source code itself.

## 15. Revision summary

| Function / mode | Core idea |
|---|---|
| `os.getcwd()` | Get the current working directory |
| `os.listdir()` | List entries in a directory |
| `os.path.join()` | Build a path using OS-appropriate separators |
| `open(path, "r")` | Open an existing file for reading |
| `read()` | Read remaining content as one string |
| `readline()` | Read the next line |
| `readlines()` | Read remaining lines into a list |
| `write()` | Write a string |
| `writelines()` | Write strings from an iterable; does not add newlines |
| `close()` | Close the file |
| `tell()` | Get the current cursor position |
| `seek(offset)` | Move the cursor |
| `truncate()` | Remove file content from a position onward |
| `r` | Read |
| `w` | Write; clears existing content |
| `a` | Append |
| `r+` | Read and write an existing file |
| `w+` | Read and write, but clears/creates first |
| `a+` | Read and append; writes remain at EOF |

### The three read methods — quick comparison

| Method | Return value | Reads |
|---|---|---|
| `read()` | String | All remaining content |
| `readline()` | String | One next line |
| `readlines()` | List of strings | All remaining lines, separated into list entries |

## 16. Dry-run practice

### Dry run 1 — cursor after `read()`

Assume a file contains `ABC`.

```python
f = open(path, "r")
x = f.read()
y = f.read()
```

| Statement | `x` | `y` | Cursor |
|---|---|---|---|
| Open file | — | — | Beginning |
| First `read()` | `"ABC"` | — | EOF |
| Second `read()` | `"ABC"` | `""` | EOF |

### Dry run 2 — line-by-line reading

Assume a file contains three lines: `A`, `B`, and `C`.

| Call | Returned value (conceptually) | Cursor after call |
|---|---|---|
| `readline()` | `"A\n"` | Start of line B |
| `readline()` | `"B\n"` | Start of line C |
| `readline()` | `"C"` (if no final newline) | EOF |
| `readline()` | `""` | EOF |

### Dry run 3 — append mode

If a file contains `ABC` and is opened with `a+`, the cursor starts at EOF. A `write("D")` appends `D`, resulting in `ABCD`. Calling `seek(0)` then allows reading from the beginning; it does not change append-mode write behaviour.

## 17. Practice exercises from the class assignment

The supplied assignment PDF contains ten exercises. They are reproduced here as practice prompts, with the supplied solution ideas in the next section.

1. **Smart Attendance Logger** — Write `mark_attendance()` that accepts names until `STOP`, stores names in a set, and reports the unique count.
2. **Restaurant Bill Splitter** — Write `split_bill(items_prices, people, tax=0.05, tip=0.10)` and return each person's share.
3. **Digital Wardrobe Organizer** — Check three requested outfit items independently against a list; print a warning for each missing item.
4. **E-Commerce Discount Engine** — Write `apply_discount(total)` using thresholds: over 500 gets 20%; over 200 gets 10%; otherwise no discount.
5. **Contact Book Search** — Search a dictionary of five contacts by name and handle a missing contact.
6. **Travel Itinerary Builder** — Use `enumerate(cities, start=1)` to print a numbered list.
7. **ATM PIN Validator** — Allow three attempts to enter a PIN; stop on success and report a blocked card after failures.
8. **Movie Ticket Age Checker** — Use a tuple `(0, 10, 15)` and conditional logic to select a ticket price.
9. **Grocery Inventory Update** — Add 10 to each stock value in a dictionary using a loop.
10. **Flexible Event Invitation** — Write `create_invite(event_name, *guest_names, **details)` and print guests and event details.

## 18. Assignment solution reference

The following solutions are based on the supplied `04-10-2026_assign_sol.pdf`. They are included as reference solutions, not as topics newly taught in this class. Some exercises use concepts from earlier classes (sets, dictionaries, tuples, functions, loops, `enumerate`, `*args`, and `**kwargs`).

In [ ]:
# 1. Smart Attendance Logger
def mark_attendance():
    attendance_set = set()

    while True:
        name = input("Enter student name (or type 'STOP' to finish): ").strip()

        if name.upper() == "STOP":
            break

        if name:
            attendance_set.add(name.title())

    print("\n--- Attendance Report ---")
    print(f"Total Unique Students: {len(attendance_set)}")
    print(f"Attendees: {', '.join(attendance_set)}")

# mark_attendance()

In [ ]:
# 2. Restaurant Bill Splitter
def split_bill(items_prices, people, tax=0.05, tip=0.10):
    subtotal = sum(items_prices)
    total_with_tax_tip = subtotal * (1 + tax + tip)
    share = total_with_tax_tip / people
    return round(share, 2)

# bill = [25.50, 40.00, 15.75]
# print(f"Each person owes: ${split_bill(bill, 3)}")

In [ ]:
# 3. Digital Wardrobe Organizer
wardrobe = ["Blue Jeans", "White Shirt", "Black Jacket"]

item1 = input("Outfit Item 1: ").title()
item2 = input("Outfit Item 2: ").title()
item3 = input("Outfit Item 3: ").title()

print("\n--- Availability Check ---")

if item1 not in wardrobe:
    print(f"Warning: {item1} is missing!")

if item2 not in wardrobe:
    print(f"Warning: {item2} is missing!")

if item3 not in wardrobe:
    print(f"Warning: {item3} is missing!")

In [ ]:
# 4. E-Commerce Discount Engine
def apply_discount(total):
    if total > 500:
        discount_rate = 0.20
    elif total > 200:
        discount_rate = 0.10
    else:
        discount_rate = 0.0

    discount_amount = total * discount_rate
    return total - discount_amount

# print(apply_discount(350))

In [ ]:
# 5. Contact Book Search
phonebook = {
    "Alice": "555-0101",
    "Bob": "555-0102",
    "Charlie": "555-0103",
    "David": "555-0104",
    "Eve": "555-0105"
}

name_query = input("Search for name: ").title()

if name_query in phonebook:
    print(f"Found: {name_query}'s number is {phonebook[name_query]}")
else:
    print("Contact not found in phonebook.")

In [ ]:
# 6. Travel Itinerary Builder
cities = ["Tokyo", "Paris", "New York", "London"]

print("--- 2026 Travel Itinerary ---")
for count, city in enumerate(cities, start=1):
    print(f"Stop {count}: {city}")

In [ ]:
# 7. ATM PIN Validator
def validate_pin():
    correct_pin = "1234"
    attempts = 0

    while attempts < 3:
        user_pin = input(f"Enter PIN ({3 - attempts} tries left): ")

        if user_pin == correct_pin:
            print("Access Granted. Welcome back!")
            return

        attempts += 1
        print("Incorrect PIN.")

    print("Card Blocked. Please contact your bank.")

# validate_pin()

In [ ]:
# 8. Movie Ticket Age Checker
PRICES = (0, 10, 15)  # Index: 0 = toddler, 1 = child, 2 = adult

age = int(input("Enter Age: "))

if age < 5:
    price = PRICES[0]
elif age < 18:
    price = PRICES[1]
else:
    price = PRICES[2]

print(f"Your ticket price is: ${price}")

In [ ]:
# 9. Grocery Inventory Update
inventory = {"Apples": 50, "Bananas": 20, "Oranges": 35}

for item in inventory:
    inventory[item] += 10

print("--- Inventory Updated ---")
for fruit, stock in inventory.items():
    print(f"{fruit}: {stock} units")

In [ ]:
# 10. Flexible Event Invitation
def create_invite(event_name, *guest_names, **details):
    print(f"\n--- {event_name.upper()} ---")
    print("Guest List:")

    for guest in guest_names:
        print(f" • {guest}")

    print("\nEvent Details:")
    for key, value in details.items():
        formatted_key = key.replace("_", " ").title()
        print(f" {formatted_key}: {value}")

# create_invite(
#     "Python Workshop",
#     "Alice",
#     "Bob",
#     date="Jan 24, 2026",
#     location="Room 404",
#     time="10 AM"
# )

## 19. Source coverage and scope notes

### Included from the class notebook and recording
- `os` import and standard-library explanation.
- `os.getcwd()`, `os.listdir()`, path separators, and `os.path.join()`.
- File object attributes: `name`, `mode`, and `closed`.
- `read()`, `readline()`, `readlines()`, EOF behaviour, and cursor movement.
- Reading lines with a `while` loop.
- `\n` and the extra newline effect of `print()`.
- Extracting bracketed content with both `find()`/slicing and `split()`.
- Writing with `w`, `write()`, and `writelines()`.
- Modes `r`, `w`, `a`, `r+`, `w+`, and `a+`.
- `tell()`, `seek()`, and `truncate()`.
- Replacing part of a file by reading, slicing, seeking, truncating, and writing.
- Reading comma-separated integer values from `variables.txt` and wrapping the logic in a function.
- The recording's practical discussion of prompts, data files, logs, and generated configuration.
- All ten assignment prompts and the supplied solution-PDF approaches.

### Important boundaries
- The supplied notes PDF did not yield extractable text; the notebook and recording were used for the lesson content.
- The original notebook contains machine-specific Windows paths. This master notebook uses `os.getcwd()` and `os.path.join()` in examples so they can be adapted to the current working directory.
- The original class files (`a.txt`, `b.txt`, etc.) are examples. The code cells that refer to them assume those files are present in the notebook's working directory.
- The assignment solutions are reference material from the supplied solution PDF, not a claim that every underlying concept was taught in this session.
- The class discussion briefly mentioned loading CSV data with a data library later in the course, but this session's practical file examples use plain text and Python's built-in file operations.